# Row softmax: one block per row

Softmax over the last dimension shows up in every attention layer and every classifier head:

$$\mathrm{softmax}(x)_{r,j} = \frac{e^{x_{r,j} - m_r}}{\sum_k e^{x_{r,k} - m_r}}, \qquad m_r = \max_k x_{r,k}$$

Subtracting the row max $m_r$ changes nothing mathematically but keeps every exponent $\le 0$, so nothing overflows.

## The mapping
Assign **one block of 64 threads per row** (grid = `rows` blocks). Inside a block, the threads cooperate:
1. Each thread scans columns `tid, tid + 64, tid + 128, ...` and keeps a running max. The 64 partial maxima go into a `float32` shared-memory array and are combined with a **tree reduction** (halve the active threads each step, `cuda.syncthreads()` between steps). Thread 0's slot then holds $m_r$.
2. Same pattern for the sum of $e^{x - m_r}$.
3. Every thread writes its columns of the output.

## Inputs and outputs
- `x`: `float32` array of shape `(rows, cols)`; `rows` may be 0, `cols` may be any positive number (smaller or larger than 64). Entries may be `-inf` (masked positions), but every row has at least one finite entry. Values up to $\pm 10^4$.
- Return `softmax(x)` as a `float32` array of shape `(rows, cols)`. For `rows == 0`, return an empty `(0, cols)` array without launching.
- Compared with relative tolerance $10^{-5}$, absolute $10^{-6}$.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import math
import numpy as np
from numba import cuda, float32

TPB = 64  # threads per block = threads per row

def softmax(x):
    """x: float32 (rows, cols). Returns the row-wise softmax, float32 (rows, cols)."""
    rows, cols = x.shape
    if rows == 0:
        return np.zeros((0, cols), dtype=np.float32)
    d_x = cuda.to_device(np.ascontiguousarray(x))
    d_out = cuda.device_array((rows, cols), dtype=np.float32)
    softmax_kernel[rows, TPB](d_x, d_out)
    return d_out.copy_to_host()

@cuda.jit
def softmax_kernel(x, out):
    row = cuda.blockIdx.x
    tid = cuda.threadIdx.x
    buf = cuda.shared.array(TPB, float32)
    # TODO 1: block-wide max of x[row, :]
    # TODO 2: block-wide sum of exp(x[row, :] - max)
    # TODO 3: write out[row, j] for this thread's columns
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]], "dtype": "float32"}], "e": {"__nd__": [[0.09003056585788727, 0.2447284460067749, 0.6652408838272095], [0.3333333432674408, 0.3333333432674408, 0.3333333432674408]], "dtype": "float32"}}, {"a": [{"__nd__": [[1000.0, 1001.0, 1002.0, 999.0]], "dtype": "float32"}], "e": {"__nd__": [[0.08714431524276733, 0.23688280582427979, 0.6439142227172852, 0.03205860033631325]], "dtype": "float32"}}, {"a": [{"__nd__": [[-3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0], [-3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0, -3.0]], "dtype": "float32"}], "e": {"__nd__": [[0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935], [0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935, 0.014285714365541935]], "dtype": "float32"}}, {"a": [{"__nd__": [[-10000.0, 0.0, 10000.0]], "dtype": "float32"}], "e": {"__nd__": [[0.0, 0.0, 1.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.8028369545936584, 0.2428499013185501, -1.6563454866409302, 0.6561048626899719, 1.1434530019760132, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}], [0.02812933921813965, 1.0524890422821045, -0.18313083052635193, -0.7620472311973572, -0.9699016213417053, -0.22172613441944122, -0.7497392892837524, 1.8806322813034058, -1.178564429283142, -1.0815995931625366, -0.30589231848716736, 0.6500137448310852, -0.26809805631637573, -0.9579081535339355, -0.8069003820419312, -0.3507920205593109, 0.9733627438545227, -0.9779819846153259, -0.0913277417421341, -0.6130539178848267, 0.3246200382709503, 0.39713579416275024, -1.4652454853057861, 2.386862277984619, 0.18960340321063995, -0.5093321800231934, -0.8939904570579529, -0.25951024889945984, 0.03181672841310501, 0.5951583981513977, 0.005621074698865414, 0.4839489161968231, -1.6099512577056885, -0.7523840665817261, -1.006557583808899, 0.16458679735660553, 0.4579990804195404, -1.4660862684249878, -0.16792866587638855, 0.7254607081413269], [{"__f__": "-inf"}, -1.115391492843628, 0.5629229545593262, {"__f__": "-inf"}, -0.43592533469200134, 1.9630240201950073, {"__f__": "-inf"}, -0.27737370133399963, 0.45479732751846313, {"__f__": "-inf"}, -0.17124268412590027, 1.3768893480300903, {"__f__": "-inf"}, 0.9126731157302856, 1.001645565032959, {"__f__": "-inf"}, -1.1647955179214478, 2.1877810955047607, {"__f__": "-inf"}, 2.229274034500122, -0.1271059513092041, {"__f__": "-inf"}, -0.16915318369865417, 0.9581159353256226, {"__f__": "-inf"}, 1.8432035446166992, 0.5350283980369568, {"__f__": "-inf"}, -0.5700718760490417, 0.28055357933044434, {"__f__": "-inf"}, 1.899982213973999, 0.9361155033111572, {"__f__": "-inf"}, 1.529097318649292, -2.373178243637085, {"__f__": "-inf"}, -1.8859753608703613, 0.0007678135298192501, {"__f__": "-inf"}]], "dtype": "float32"}], "e": {"__nd__": [[0.06420406699180603, 0.18268321454524994, 0.027345670387148857, 0.27616778016090393, 0.4495992660522461, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.019456977024674416, 0.05419374257326126, 0.015751659870147705, 0.008828888647258282, 0.007171930279582739, 0.015155299566686153, 0.008938226848840714, 0.1240529716014862, 0.0058212317526340485, 0.006413958966732025, 0.0139319421723485, 0.03623734042048454, 0.014468563720583916, 0.007258464116603136, 0.008441634476184845, 0.013320237398147583, 0.050070855766534805, 0.007114211097359657, 0.0172661654651165, 0.010247386060655117, 0.02617216482758522, 0.028140567243099213, 0.004370296839624643, 0.20580697059631348, 0.02286665514111519, 0.011367338709533215, 0.007737556006759405, 0.014593353495001793, 0.019528856500983238, 0.03430306538939476, 0.019023925065994263, 0.03069271519780159, 0.0037815168034285307, 0.008914616890251637, 0.006913796998560429, 0.022301705554127693, 0.029906490817666054, 0.004366623237729073, 0.015992945060133934, 0.03907712176442146], [0.0, 0.004678327590227127, 0.02505955472588539, 0.0, 0.00922951940447092, 0.1016317754983902, 0.0, 0.010815263725817204, 0.022491319105029106, 0.0, 0.012026222422719002, 0.05655544996261597, 0.0, 0.035552315413951874, 0.03886047750711441, 0.0, 0.004452815279364586, 0.12724487483501434, 0.0, 0.132635697722435, 0.012568909674882889, 0.0, 0.012051377445459366, 0.03720518574118614, 0.0, 0.09015548229217529, 0.024370191618800163, 0.0, 0.008070861920714378, 0.018894782289862633, 0.0, 0.09542250633239746, 0.0363955944776535, 0.0, 0.0658533051609993, 0.0013299668207764626, 0.0, 0.0021648621186614037, 0.01428342703729868, 0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.007729627657681704], [-1.0980663299560547], [1.2332699298858643], [-1.7795740365982056]], "dtype": "float32"}], "e": {"__nd__": [[1.0], [1.0], [1.0], [1.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[-4.380573749542236, 4.151710510253906, -2.5948376655578613, 0.739955484867096, 2.4295849800109863, -0.3194476366043091, 0.8294945955276489, 0.3619106113910675, 0.642433762550354, 3.561150074005127, 2.802654504776001, 0.9704408645629883, 2.752406120300293, -1.6252925395965576, -2.2929816246032715, -3.0050971508026123, 2.3524019718170166, 4.874198913574219, 0.25500085949897766, -5.935834884643555, -1.526105284690857, 1.4039983749389648, 0.8910630345344543, 0.2811436355113983, 2.9664385318756104, 2.7798917293548584, -4.022326946258545, 1.3040817975997925, -2.218905210494995, 0.6984953284263611, 3.7317593097686768, 5.494124412536621, 2.0999932289123535, -1.5168590545654297, -1.0537772178649902, 1.599217414855957, -2.96382737159729, -0.20297940075397491, 1.8147975206375122, 2.7554972171783447, -0.8911890983581543, 1.1093757152557373, 4.087569713592529, 0.9157546758651733, -5.149123668670654, 0.6429758071899414, 0.7542873620986938, -1.264284610748291, 0.12894514203071594, -4.42891263961792, 3.73828125, 1.264676570892334, -1.6305745840072632, 0.7068389654159546, 0.3974117934703827, -2.5911262035369873, -8.00154972076416, -8.542325973510742, 6.065435886383057, -3.3576393127441406, -0.8832419514656067, 1.7146072387695312, -1.0853239297866821, 3.5750904083251953, -2.846008777618408, 1.6886543035507202, -6.261288642883301, 2.144368886947632, 2.063114881515503, 1.1008825302124023, 0.9035385251045227, 0.8048731684684753, 2.5923259258270264, 2.8873579502105713, -2.8331472873687744, 3.1724939346313477, -2.8427045345306396, 0.49384260177612305, -0.7581192851066589, 1.4417060613632202, -0.9958535432815552, 1.214633822441101, -1.4365570545196533, -2.8762388229370117, 3.203350305557251, 1.7176460027694702, 1.5418415069580078, 2.8695380687713623, -3.1792023181915283, 2.7432830333709717, -3.16972017288208, 1.1422377824783325, -0.5363333821296692, 0.8154827356338501, 2.092130422592163, 0.06404080241918564, -2.7618722915649414, 2.789047956466675, 3.5313949584960938, 2.055518865585327, 3.1101677417755127, -5.289275169372559, 0.21634921431541443, -0.342312216758728, -5.862587928771973, 0.3261885643005371, 0.009925948455929756, -4.519138813018799, -2.519913911819458, 0.2552332878112793, 1.8055520057678223, 2.122699022293091, -3.3161330223083496, -3.2308297157287598, -1.516455888748169, -0.14977487921714783, -1.7251774072647095, -0.5757755637168884, 2.12919282913208, 2.053377151489258, 0.4143298268318176, -3.4237897396087646, -1.4730463027954102, 1.6913535594940186, 2.610595703125, 3.186797618865967, -3.1834044456481934, -1.392866611480713, -3.630115509033203, -8.824305534362793, -3.275932788848877, -5.312815189361572, 4.181337833404541, 1.4225255250930786, -1.9238873720169067, 0.20257234573364258, 1.247017741203308, 5.255002498626709, 2.33528733253479, 4.584023475646973, 1.4658737182617188, 3.9184410572052, 4.343286514282227, 2.808608055114746, 0.11270732432603836, 3.3018343448638916, -1.1622834205627441, 4.303391933441162, -0.8023597002029419, -2.0215001106262207, 5.674285888671875, -1.21164071559906, 0.5110604166984558, -3.1692497730255127, -2.0771517753601074, -0.9961025714874268, -2.944645881652832, 3.684387445449829, -2.1508359909057617, -3.7450740337371826, 3.366257905960083, 2.1453781127929688, -6.954225540161133, -1.6846601963043213, 3.009143352508545, -2.282008647918701, -0.8106915354728699, -0.6951363682746887, -6.748411655426025, -2.636309862136841, 2.335693836212158, 4.379350185394287, -0.19197416305541992, -0.27866679430007935, 1.7096095085144043, 0.5522719621658325, 1.5820419788360596, -2.5679075717926025, 0.12596382200717926, 1.5734106302261353, 1.8488223552703857, -4.612297058105469, 2.523946762084961, 0.40747520327568054, 0.5282355546951294, -5.248728275299072, 0.17875085771083832, -4.680682182312012, -7.509388446807861, 0.3496165871620178, 0.05434668809175491, 0.24313896894454956, 0.6023680567741394, 0.07784158736467361, 3.3829898834228516, 2.5185728073120117, 4.971261024475098, 7.680352210998535, 2.4279868602752686, 2.5165152549743652], [5.820871353149414, -2.491368055343628, -0.8450035452842712, 0.9003818035125732, -4.535930633544922, 3.0340969562530518, 4.873140335083008, -3.6429800987243652, -2.311173439025879, 2.8306679725646973, 4.17910623550415, -4.85902738571167, 1.7280349731445312, -0.5315898656845093, -5.469864368438721, -1.2077152729034424, -1.6556761264801025, 0.3601556718349457, 0.14672420918941498, -0.7954673767089844, -0.3771107792854309, -0.4494410753250122, 1.557759404182434, 2.9125542640686035, 2.8939247131347656, -2.6653454303741455, -2.3716483116149902, -2.0096306800842285, 4.268610000610352, 1.4353008270263672, -2.3007328510284424, -3.4736812114715576, -3.0213029384613037, 0.08520744740962982, -3.312601089477539, -0.023867934942245483, 4.0423665046691895, -0.7697768211364746, 2.563230037689209, -4.5654072761535645, 1.067893624305725, 0.873055100440979, -2.77018666267395, 4.136168956756592, -0.02081591822206974, -2.067342519760132, -0.8823287487030029, 2.856527090072632, -4.1215128898620605, 0.4458247423171997, -2.8857202529907227, 0.6794872283935547, -1.587632656097412, -0.2219233214855194, 2.2806291580200195, 0.12616901099681854, 1.6546316146850586, 2.0960533618927, 1.2206107378005981, 0.6781290769577026, 0.12740223109722137, 1.9123457670211792, 3.613327741622925, 5.52505350112915, 5.280681133270264, -2.9791390895843506, 0.8812757730484009, 1.4675390720367432, 3.647148370742798, 3.190869092941284, -4.295556545257568, 0.8268079161643982, -5.131965160369873, -1.4239883422851562, -2.7811179161071777, -1.8371118307113647, 4.744601726531982, -1.6294546127319336, -1.501682996749878, 0.3678668439388275, 3.212114095687866, -5.108206748962402, 4.874716758728027, -2.2953834533691406, -2.9863836765289307, 0.37886321544647217, 2.0138909816741943, 8.115957260131836, 2.6842198371887207, -1.7894116640090942, -0.6296955347061157, -2.8861606121063232, 1.9518827199935913, 2.467430353164673, -1.020476222038269, 3.3780064582824707, -5.089317798614502, -1.4609907865524292, -2.8007349967956543, -1.798766016960144, -4.939969539642334, 2.834238290786743, -2.8941423892974854, -0.4329959452152252, 3.515864849090576, 4.792393207550049, 3.5111842155456543, -6.737178325653076, -0.9492468237876892, -1.0925521850585938, -2.0932059288024902, -2.395916700363159, -1.7004507780075073, 1.7250888347625732, -1.4278193712234497, -4.083128929138184, -4.782498836517334, 0.39304476976394653, -5.214312553405762, 2.069927453994751, -1.5259653329849243, -1.5524245500564575, 0.656833291053772, -1.1122124195098877, 2.104970693588257, -3.8888051509857178, -5.48175048828125, 5.883237361907959, 0.4920009970664978, -7.778185844421387, 5.093276500701904, -4.270853519439697, 6.168915748596191, 5.45492696762085, 2.642653465270996, 0.00502935191616416, -2.6286227703094482, 2.8721084594726562, -1.2032558917999268, 3.27331805229187, -0.839617133140564, -4.745075702667236, -2.109840154647827, 0.1899542510509491, -1.4753152132034302, 0.4297836124897003, -2.7743163108825684, 4.878126621246338, -1.565898060798645, 1.5865414142608643, -5.024765968322754, 2.5203158855438232, 4.687751293182373, 1.84561288356781, -0.7029129862785339, -2.7624435424804688, 1.8656314611434937, 1.7882349491119385, -3.8854455947875977, 0.09306485950946808, 1.6796457767486572, -2.3039815425872803, 1.8772306442260742, -0.7665151357650757, 4.425693511962891, -3.781264305114746, 1.6260946989059448, 0.4068199098110199, -5.604914665222168, -2.756223678588867, -2.5329296588897705, 2.8513126373291016, -1.0356727838516235, 0.019037554040551186, 0.49155351519584656, 7.818023204803467, -0.6782389879226685, 5.869343280792236, -0.5241873264312744, -4.715814113616943, -2.4239730834960938, -3.713181257247925, -1.085877537727356, -4.376965522766113, -2.7261359691619873, 5.784425258636475, 0.7482319474220276, 1.0338013172149658, 5.501018047332764, -4.946911811828613, 1.729135513305664, -1.3950730562210083, -0.07293348759412766, -2.0317440032958984, 0.3809589743614197, 3.740372657775879, 3.647589683532715, 0.7697542905807495, 2.4978060722351074, -3.5668222904205322], [-8.776464462280273, 3.159550189971924, 6.1408915519714355, 1.4311294555664062, -1.8296304941177368, 0.16933786869049072, 3.698716402053833, -3.4903969764709473, 0.24312102794647217, 1.8695638179779053, -3.0692434310913086, 6.3166303634643555, 2.4725492000579834, 5.021108627319336, -3.2228972911834717, 0.6742726564407349, 4.630692481994629, 1.3753561973571777, 3.815760374069214, -5.54647970199585, -0.24612821638584137, -5.986801624298096, -1.4893616437911987, 0.8876249194145203, -0.3241492807865143, -3.7755305767059326, 0.9075388312339783, 1.978611946105957, -0.28089234232902527, 2.45143985748291, 3.443631649017334, 3.74157452583313, -1.0072236061096191, 1.5586689710617065, -1.9074602127075195, -0.2536880075931549, 3.2275640964508057, -0.31549298763275146, -1.3561139106750488, -1.7830042839050293, 1.5570030212402344, 4.849304676055908, 2.7813568115234375, -0.6708593964576721, 2.408538818359375, 2.8568854331970215, 4.402449607849121, -2.025803804397583, 1.9138283729553223, -1.2008954286575317, -1.628896951675415, 3.7692723274230957, 1.642708420753479, 0.23053178191184998, 4.084079265594482, 0.5032937526702881, -2.649411201477051, 2.7305729389190674, -2.7767586708068848, -2.6052422523498535, -1.2479851245880127, -0.4696873724460602, 2.698638439178467, -1.0626109838485718, -5.454834938049316, -1.7015588283538818, 0.9428650140762329, -0.360907644033432, -0.3624976575374603, -3.6560957431793213, -1.1666220426559448, 1.7376583814620972, 1.7896870374679565, -2.4994635581970215, -2.1736929416656494, -2.907715320587158, 2.8033294677734375, -2.2449300289154053, -0.8157317042350769, -2.4688589572906494, -2.7280962467193604, -0.3895658552646637, 2.693272590637207, -1.723044753074646, 3.5132646560668945, -0.1660425215959549, 2.1692748069763184, 7.437073707580566, 6.8064727783203125, 0.8685697913169861, 0.24017634987831116, -0.44793373346328735, 2.137800931930542, -5.607584476470947, -2.107691764831543, -0.40433022379875183, -2.2881381511688232, -2.9791085720062256, 3.8924970626831055, -4.375142574310303, -1.2913044691085815, -1.7398754358291626, -1.4040122032165527, -3.1387012004852295, 2.287475824356079, -0.34631821513175964, 5.226205825805664, -4.093614101409912, 0.3676208555698395, -3.8407070636749268, 0.6891486048698425, 3.50064754486084, -3.424987554550171, 2.969010353088379, -2.4476988315582275, -3.436281681060791, 0.979140043258667, 2.2385895252227783, 0.39924490451812744, 0.935003399848938, -0.10046908259391785, -1.5253665447235107, -6.450495719909668, -1.1060553789138794, -2.1112756729125977, 1.6512115001678467, -2.4830429553985596, 6.04143762588501, 1.8600342273712158, -1.2644625902175903, 1.9710063934326172, -0.7417924404144287, 3.866731643676758, -1.2629867792129517, 4.317052364349365, -3.191295862197876, 0.33028507232666016, -0.18325914442539215, -2.6816790103912354, -0.9531489610671997, 0.05101713165640831, -3.55303692817688, -2.5437989234924316, 1.4807971715927124, -2.287266492843628, 4.769752025604248, 4.7624335289001465, -1.3954734802246094, 1.0870020389556885, 1.6146646738052368, 2.1775786876678467, -1.5882039070129395, 2.248678684234619, -1.5808006525039673, -0.2795950770378113, 3.7525148391723633, -0.23574665188789368, -2.879741907119751, -1.7082016468048096, 0.9748538136482239, -2.27657151222229, -2.9073870182037354, 0.4600085914134979, -2.825439929962158, 6.309759616851807, -2.0001156330108643, 2.078397750854492, -4.381831645965576, -0.9966022968292236, 2.4723775386810303, 1.6192182302474976, 1.9361472129821777, -1.716636061668396, 0.5863001942634583, -0.589593231678009, -4.223803520202637, 3.4364430904388428, -2.0309081077575684, -0.6951362490653992, 2.581300973892212, 1.9826598167419434, 7.7976508140563965, -0.20846758782863617, 4.337088108062744, -0.30674606561660767, 0.10861923545598984, 3.2549471855163574, 2.016481637954712, -1.483315110206604, -0.22946082055568695, 0.5669417381286621, 5.096640586853027, -2.2595341205596924, 0.7136101126670837, 1.8550114631652832, 0.44665780663490295, -1.4890614748001099, 2.537503480911255, -2.7528560161590576, 5.8504180908203125], [-7.843629837036133, -5.803753852844238, 0.5180327296257019, 1.9616109132766724, 1.4253370761871338, -1.7304725646972656, -0.4068010747432709, 0.10476049780845642, -2.614374876022339, -1.0939762592315674, -2.808662176132202, 3.5271735191345215, 1.297882318496704, -1.2589516639709473, -1.08138906955719, -1.324073076248169, -0.9022588133811951, 0.854422390460968, -1.02799654006958, -1.4828765392303467, 1.190161108970642, -0.45106568932533264, 0.046113915741443634, 0.2714543342590332, -0.5084335803985596, -3.3187313079833984, -0.6949946284294128, 3.6534554958343506, -2.51884388923645, -2.4263346195220947, -1.2236759662628174, -1.111161470413208, 4.195183277130127, -0.8662658929824829, 3.511596202850342, 2.4287590980529785, 2.217337131500244, 1.9471205472946167, 1.1027525663375854, 1.6167397499084473, 3.691709041595459, -6.07276725769043, 1.1200200319290161, 3.540923833847046, -1.9256335496902466, -0.09801365435123444, 2.6098968982696533, -5.243622303009033, 0.9139586687088013, -3.260427236557007, 0.7144058346748352, 1.1363868713378906, 0.7449356913566589, -3.0271189212799072, 0.03287431597709656, -1.8913838863372803, 0.9274619221687317, -1.1167067289352417, -2.522317886352539, 2.8255248069763184, -3.651728868484497, -1.5121877193450928, 3.2682723999023438, 1.0976030826568604, 2.62162184715271, -2.9882068634033203, 0.5241443514823914, -2.6405797004699707, -5.312527656555176, 1.7505613565444946, -3.0956366062164307, 0.33154499530792236, -2.865550994873047, 1.64694344997406, 1.7121107578277588, -3.0608973503112793, -3.0730605125427246, -7.997745037078857, -1.4365755319595337, -0.7375108599662781, -2.244093656539917, 2.065338611602783, -0.28398653864860535, -0.025082534179091454, -1.9162436723709106, -4.568145751953125, 0.13838298618793488, 2.50469970703125, 3.6108405590057373, 0.4950502812862396, -3.8203697204589844, -1.0429521799087524, 0.42660412192344666, -3.351724147796631, -1.1646170616149902, -1.9779572486877441, -2.473877191543579, -3.1215016841888428, 0.8954185843467712, 1.5877575874328613, 3.656508445739746, -0.8318489789962769, 0.1553916484117508, -5.23064661026001, 0.8125642538070679, 4.074106693267822, -2.3762600421905518, 1.4583483934402466, -1.601324439048767, 1.5294163227081299, 2.149749755859375, -4.382948875427246, -2.7337491512298584, -2.2737319469451904, -0.6464659571647644, 3.90248441696167, 0.09343104809522629, 0.3721645176410675, 0.5425887107849121, -1.9857147932052612, 3.223085403442383, 1.52252995967865, 2.3332605361938477, 0.750093936920166, -4.147594451904297, 1.5873652696609497, -1.9174004793167114, -2.762976884841919, 8.069451332092285, 2.7459897994995117, -1.26760995388031, 2.4781911373138428, 3.311445713043213, 0.6678952574729919, -1.2126729488372803, 2.94891357421875, 2.2755417823791504, 2.0610406398773193, -5.042871952056885, 1.312956690788269, 6.193021297454834, -3.285369873046875, -1.241458773612976, -2.399533748626709, -1.5441203117370605, -0.47408169507980347, 6.078917980194092, -0.9345952272415161, 4.8213791847229, -2.8847806453704834, 1.8980776071548462, -0.17118586599826813, -1.2811273336410522, 0.842583179473877, 4.045777797698975, 1.1979132890701294, 2.0069265365600586, 0.2639584243297577, 1.481913447380066, 1.2632864713668823, 2.5289902687072754, -0.33310559391975403, -1.7514697313308716, -1.3649030923843384, -1.422653079032898, 1.6892399787902832, -2.362301826477051, 2.2495815753936768, 0.804448664188385, 3.8921549320220947, 0.7104963064193726, 3.3295934200286865, 1.3971067667007446, 0.6440927386283875, -0.829745352268219, 1.5075709819793701, 0.3618265390396118, -1.8102850914001465, -1.4120986461639404, -5.7799072265625, 3.5817086696624756, 3.5199553966522217, -1.370261311531067, -3.765700578689575, 2.098691463470459, 3.4264895915985107, -0.030332718044519424, -1.9496406316757202, 3.89062762260437, 2.730952262878418, 3.155836343765259, 5.767828464508057, -0.05032505467534065, 0.5558829307556152, 6.134211540222168, 2.6032214164733887, 7.526553153991699, -7.807209491729736, -4.585002422332764, 1.7853176593780518], [1.4098224639892578, -3.466832399368286, -2.859074354171753, -2.7053442001342773, 1.8238283395767212, -4.404687404632568, -0.591260552406311, -2.4751362800598145, -9.245290756225586, 0.6393855214118958, 5.3702263832092285, -0.8229435682296753, 2.074066638946533, -0.082880899310112, -0.809847354888916, -5.254215240478516, -0.7801812291145325, -6.334954261779785, -2.1361453533172607, 0.1536731719970703, 0.2459653913974762, -2.7059364318847656, -0.7445694208145142, -3.3991498947143555, 2.0467376708984375, -3.713118553161621, 8.094853401184082, 2.338071823120117, -0.37734726071357727, 5.3027825355529785, -2.875960111618042, 3.654961347579956, -1.9800100326538086, 0.09212277084589005, -1.7153536081314087, -2.365823745727539, -1.8123059272766113, 0.5057477951049805, -0.9816524386405945, 5.489386081695557, 0.18686071038246155, -8.13375186920166, -2.360611915588379, -1.0058151483535767, 0.7495865225791931, 1.6179381608963013, -4.148590564727783, 0.3660854399204254, -2.8423774242401123, -0.03272976726293564, -1.1527893543243408, 3.446524143218994, 2.7359330654144287, 4.3017754554748535, -2.6740336418151855, -1.888871669769287, -0.2272394895553589, -2.0458343029022217, -2.2589635848999023, 1.1796162128448486, 6.277242183685303, 0.6365002989768982, 4.372213840484619, -4.645281791687012, 5.718446254730225, 6.276474952697754, 0.20620188117027283, -1.841665506362915, 1.527572751045227, -5.339926719665527, 0.5759031772613525, -5.610569953918457, 4.461696624755859, -0.433593213558197, -2.567295789718628, -6.635106563568115, 4.052151203155518, 0.01031497586518526, -1.9086084365844727, 3.32832670211792, -4.297310829162598, 3.046555995941162, 3.743302345275879, -0.9873322248458862, 3.4481966495513916, 4.953287601470947, -2.343067169189453, -2.190700054168701, 2.4024412631988525, 2.2285454273223877, -2.3822638988494873, -9.440979957580566, -4.684643268585205, 3.6566619873046875, -1.5358927249908447, -2.865441083908081, 1.083008050918579, -5.332690238952637, 6.652711391448975, -5.868588447570801, -0.6320453882217407, 1.0994261503219604, 1.808366060256958, -3.3826329708099365, -1.074510931968689, 3.0646252632141113, 0.35626938939094543, 2.003115653991699, 0.43468931317329407, 0.5594334602355957, -2.618636131286621, -3.582659959793091, -0.8566110134124756, -3.40094256401062, 4.269730091094971, 2.1309306621551514, 3.686656951904297, -4.509138107299805, -3.9543678760528564, 1.9270411729812622, 0.453176885843277, 0.6521934866905212, 2.8792636394500732, -3.0647103786468506, -3.8564541339874268, -2.6190149784088135, 6.2213945388793945, -0.2246103733778, -2.4209635257720947, 4.270516395568848, 2.6012985706329346, 2.536371946334839, 1.7876801490783691, -0.7324182391166687, 5.224324703216553, 2.4270968437194824, -3.6467528343200684, -3.479454278945923, -2.0259859561920166, 2.490983247756958, -1.9270473718643188, -0.8773730397224426, -1.4591869115829468, 0.14437925815582275, 3.6284921169281006, -1.364836573600769, -0.012385202571749687, 5.098664283752441, -4.646077632904053, 0.9290333986282349, 0.5634783506393433, -0.23068010807037354, 3.0321531295776367, 6.547879695892334, 0.0311213880777359, -0.11996018141508102, 1.9345049858093262, -3.5040438175201416, -1.0481489896774292, 0.9473658204078674, 0.6742525696754456, -2.1782350540161133, 3.306030511856079, -3.38527512550354, -2.071207046508789, -2.9982783794403076, 1.1769208908081055, 2.4462473392486572, -0.2868155539035797, 3.4872028827667236, -1.2095870971679688, 0.3064044713973999, 0.6822541952133179, -0.07270034402608871, 0.5603883862495422, -3.017610788345337, -6.223381996154785, -1.6105605363845825, -1.1033660173416138, -0.6873399019241333, -3.0965969562530518, -3.7240347862243652, 6.038151741027832, 2.2650094032287598, 1.023123025894165, 0.5213441252708435, -1.8802822828292847, -0.3894772529602051, 3.4364142417907715, 3.0262391567230225, -1.6706076860427856, -1.4295357465744019, -1.544835090637207, -2.0849151611328125, 2.766291856765747, 1.9582947492599487, 0.974402904510498, -1.768155574798584, 3.7870123386383057, 0.10102920234203339]], "dtype": "float32"}], "e": {"__nd__": [[2.437009925415623e-06, 0.012370334938168526, 1.4534250112774316e-05, 0.00040801402064971626, 0.002210404956713319, 0.00014144324813969433, 0.00044623270514421165, 0.00027957119164057076, 0.00037010241067036986, 0.006853372324258089, 0.00320991687476635, 0.0005137757398188114, 0.003052609972655773, 3.8323116314131767e-05, 1.9655652067740448e-05, 9.643158591643441e-06, 0.0020462165120989084, 0.025477346032857895, 0.0002512245555408299, 5.145364525560581e-07, 4.2319210479035974e-05, 0.0007926204125396907, 0.0004745700571220368, 0.0002578787971287966, 0.0037811531219631433, 0.0031376772094517946, 3.486922651063651e-06, 0.000717252551112324, 2.116694304277189e-05, 0.00039144360926002264, 0.008128286339342594, 0.04735702648758888, 0.0015897615812718868, 4.271230136509985e-05, 6.786832091165707e-05, 0.0009634913294576108, 1.0049460797745269e-05, 0.00015891452494543046, 0.0011952893109992146, 0.003062059637159109, 7.98506080172956e-05, 0.00059035379672423, 0.01160180289298296, 0.0004864338261540979, 1.1300060123176081e-06, 0.0003703031106851995, 0.00041390370461158454, 5.4985030146781355e-05, 0.00022147093841340393, 2.3220115963340504e-06, 0.00818147137761116, 0.0006895386031828821, 3.8121248508105054e-05, 0.0003947233490180224, 0.0002896744990721345, 1.4588297744921874e-05, 6.520600948078936e-08, 3.796916558940211e-08, 0.08384976536035538, 6.778167517040856e-06, 8.048770541790873e-05, 0.0010813367553055286, 6.576065788976848e-05, 0.006949579808861017, 1.1306036867608782e-05, 0.001053634099662304, 3.715980483320891e-07, 0.0016618974041193724, 0.00153220200445503, 0.0005853609181940556, 0.0004805276694241911, 0.00043537994497455657, 0.00260105449706316, 0.003493655240163207, 1.1452396392996889e-05, 0.004646362736821175, 1.134345893660793e-05, 0.0003189992276020348, 9.121565381065011e-05, 0.0008230790845118463, 7.191555778263137e-05, 0.0006558814202435315, 4.628367241821252e-05, 1.0969373761327006e-05, 0.004791966639459133, 0.001084627816453576, 0.0009097661823034286, 0.003431951627135277, 8.102267202048097e-06, 0.003024887992069125, 8.179454198398162e-06, 0.0006100761238485575, 0.00011386493861209601, 0.00044002372305840254, 0.0015773108461871743, 0.0002075529919238761, 1.229845656780526e-05, 0.003166538430377841, 0.006652453914284706, 0.0015206073876470327, 0.004365611355751753, 9.822306310525164e-07, 0.000241699512116611, 0.00013824593042954803, 5.536393246075022e-07, 0.0002697605232242495, 0.00019661980331875384, 2.121676971000852e-06, 1.5665054888813756e-05, 0.0002512828796170652, 0.0011842892272397876, 0.0016262708231806755, 7.065427325869678e-06, 7.694578016526066e-06, 4.272953810868785e-05, 0.00016759846766944975, 3.468017894192599e-05, 0.00010946128895739093, 0.0016368662472814322, 0.0015173538122326136, 0.0002946169697679579, 6.344295798044186e-06, 4.462524884729646e-05, 0.001056482084095478, 0.0026490113232284784, 0.0047133006155490875, 8.068294846452773e-06, 4.835061918129213e-05, 5.16151703777723e-06, 2.8639757232440388e-08, 7.355238722084323e-06, 9.59378553488932e-07, 0.012742319144308567, 0.0008074422366917133, 2.843039510480594e-05, 0.00023839248751755804, 0.0006774691282771528, 0.037285082042217255, 0.002011494245380163, 0.019060427322983742, 0.0008432131726294756, 0.009796565398573875, 0.014982417225837708, 0.003229085123166442, 0.0002179036964662373, 0.005287919659167528, 6.088958616601303e-05, 0.014396465383470058, 8.726822852622718e-05, 2.5786353944567963e-05, 0.05670580640435219, 5.795725155621767e-05, 0.000324539199937135, 8.183308636944275e-06, 2.43905269599054e-05, 7.189765892690048e-05, 1.0244082659482956e-05, 0.007752212230116129, 2.2657934096059762e-05, 4.6009913603484165e-06, 0.005639798939228058, 0.0016635749489068985, 1.8583808980565664e-07, 3.61141974281054e-05, 0.003946125041693449, 1.9872521079378203e-05, 8.654411067254841e-05, 9.714551561046392e-05, 2.283065470010115e-07, 1.3943819794803858e-05, 0.0020123126450926065, 0.015532598830759525, 0.00016067310934886336, 0.00014733060379512608, 0.0010759460274130106, 0.0003381934075150639, 0.0009470840450376272, 1.4930987163097598e-05, 0.0002208115765824914, 0.0009389446349814534, 0.0012366590090095997, 1.9329531824041624e-06, 0.0024291404988616705, 0.00029260426526889205, 0.00033016136148944497, 1.0228750397800468e-06, 0.00023278077424038202, 1.8051865708912374e-06, 1.0666706629081091e-07, 0.0002761551004368812, 0.0002055506920441985, 0.0002482620475348085, 0.00035556708462536335, 0.00021043731248937547, 0.005734959617257118, 0.0024161215405911207, 0.028074223548173904, 0.4215519428253174, 0.002206875942647457, 0.0024111552629619837], [0.03180135786533356, 7.807029760442674e-06, 4.050355346407741e-05, 0.000232008722377941, 1.0105171668328694e-06, 0.0019595881458371878, 0.012326826341450214, 2.4680102796992287e-06, 9.348533239972312e-06, 0.0015988834202289581, 0.0061579495668411255, 7.3151687729478e-07, 0.0005308226100169122, 5.541239806916565e-05, 3.9713819433018216e-07, 2.818184111674782e-05, 1.800621612346731e-05, 0.00013517205661628395, 0.00010919302440015599, 4.2560455767670646e-05, 6.466903141699731e-05, 6.015664985170588e-05, 0.00044771289685741067, 0.0017353200819343328, 0.0017032912001013756, 6.5603744587861e-06, 8.799932402325794e-06, 1.2638689440791495e-05, 0.0067345271818339825, 0.00039611069951206446, 9.446651347388979e-06, 2.9232951419544406e-06, 4.595555765263271e-06, 0.0001026782047119923, 3.434226755416603e-06, 9.206779941450804e-05, 0.005370946601033211, 4.366800203570165e-05, 0.0012236860347911716, 9.81165726443578e-07, 0.00027431754278950393, 0.000225754440180026, 5.907403647142928e-06, 0.005899139679968357, 9.234919707523659e-05, 1.1929927495657466e-05, 3.901959280483425e-05, 0.0016407688381150365, 1.5294061768145184e-06, 0.00014726261724717915, 5.262849299469963e-06, 0.00018602481577545404, 1.9274053556728177e-05, 7.552547322120517e-05, 0.0009224397363141179, 0.00010697146353777498, 0.0004932541050948203, 0.000766968703828752, 0.0003195786848664284, 0.0001857723545981571, 0.00010710344940889627, 0.0006382555584423244, 0.003497210331261158, 0.02365775778889656, 0.018528660759329796, 4.793464995600516e-06, 0.00022761794389225543, 0.0004090885631740093, 0.0036175092682242393, 0.002292188350111246, 1.2850991879531648e-06, 0.00021555171406362206, 5.567869152400817e-07, 2.270090044476092e-05, 5.843176040798426e-06, 1.5018479643913452e-05, 0.010839960537850857, 1.8484606698621064e-05, 2.1003932488383725e-05, 0.00013621845573652536, 0.0023414066527038813, 5.701738814423152e-07, 0.01234627515077591, 9.497317478235345e-06, 4.758866452903021e-06, 0.000137724622618407, 0.000706472375895828, 0.31563764810562134, 0.0013810684904456139, 1.5752220861031674e-05, 5.0234248192282394e-05, 5.260531452222494e-06, 0.0006639957427978516, 0.001111897174268961, 3.3984906622208655e-05, 0.0027639036998152733, 5.810465495414974e-07, 2.1876259779674e-05, 5.729667464038357e-06, 1.5605552107444964e-05, 6.746392955392366e-07, 0.001604601857252419, 5.218712431087624e-06, 6.115411088103428e-05, 0.0031724446453154087, 0.01137059647589922, 0.0031576300971210003, 1.1182883241644959e-07, 3.649394784588367e-05, 3.1621606467524543e-05, 1.16253313535708e-05, 8.588945092924405e-06, 1.721776970953215e-05, 0.0005292611313052475, 2.2614100089413114e-05, 1.5892527471805806e-06, 7.896966849330056e-07, 0.00013969169231131673, 5.127741928845353e-07, 0.0007471904973499477, 2.0500046957749873e-05, 1.99647547560744e-05, 0.0001818579767132178, 3.1005998607724905e-05, 0.0007738389540463686, 1.930130110849859e-06, 3.924455143078376e-07, 0.033847834914922714, 0.0001542221289128065, 3.9486639735741846e-08, 0.015362263657152653, 1.3172416402085219e-06, 0.045040104538202286, 0.022055556997656822, 0.0013248390750959516, 9.476712148170918e-05, 6.805764314776752e-06, 0.0016665341099724174, 2.8307795219006948e-05, 0.002489186357706785, 4.072231240570545e-05, 8.198102250389638e-07, 1.143355530075496e-05, 0.00011401695519452915, 2.1565132556133904e-05, 0.0001449192495783791, 5.883054200239712e-06, 0.012388445436954498, 1.9697563402587548e-05, 0.0004607861628755927, 6.197909669936053e-07, 0.0011722836643457413, 0.010240893810987473, 0.0005970529746264219, 4.668764813686721e-05, 5.953321760898689e-06, 0.0006091255927458405, 0.0005637594149447978, 1.936625949383597e-06, 0.00010348825890105218, 0.0005057479720562696, 9.416006832907442e-06, 0.000616231991443783, 4.381070175440982e-05, 0.007880025543272495, 2.1492708128789673e-06, 0.000479377165902406, 0.00014162925072014332, 3.4696833495218016e-07, 5.990465979266446e-06, 7.489211384381633e-06, 0.001632235012948513, 3.3472351788077503e-05, 9.610392589820549e-05, 0.00015415309462696314, 0.2343136966228485, 4.785396595252678e-05, 0.03338079899549484, 5.5824108130764216e-05, 8.441528507319163e-07, 8.351321412192192e-06, 2.3006953142612474e-06, 3.1833380489842966e-05, 1.18462628506677e-06, 6.1734431255899835e-06, 0.03066318854689598, 0.00019926285312976688, 0.00026512303156778216, 0.023095911368727684, 6.699722803205077e-07, 0.0005314071313478053, 2.3366887035081163e-05, 8.765947131905705e-05, 1.236226762557635e-05, 0.00013801355089526623, 0.003970969468355179, 0.003619106952100992, 0.00020359797053970397, 0.001146190334111452, 2.6633131255948683e-06], [1.6099704680527793e-08, 0.0024578957818448544, 0.048455554991960526, 0.0004364352789707482, 1.6741327272029594e-05, 0.00012357502419035882, 0.004214251413941383, 3.1807396680960665e-06, 0.00013303759624250233, 0.0006765959551557899, 4.846552656090353e-06, 0.057765159755945206, 0.0012365240836516023, 0.015813501551747322, 4.156251634412911e-06, 0.0002047486777883023, 0.010702184401452541, 0.00041276024421676993, 0.0047375294379889965, 4.0698901671021304e-07, 8.156352123478428e-05, 2.62031392139761e-07, 2.3527016310254112e-05, 0.0002534421219024807, 7.544177060481161e-05, 2.3916422833281104e-06, 0.0002585398033261299, 0.0007545507978647947, 7.877682219259441e-05, 0.001210695831105113, 0.0032654155511409044, 0.00439879298210144, 3.8102771213743836e-05, 0.0004958035424351692, 1.5487774362554774e-05, 8.094924851320684e-05, 0.0026308835949748755, 7.609766180394217e-05, 2.6880383302341215e-05, 1.7540392946102656e-05, 0.0004949782160110772, 0.013317248784005642, 0.0016838994342833757, 5.33380989509169e-05, 0.0011598541168496013, 0.0018160082399845123, 0.008518200367689133, 1.3759186003881041e-05, 0.0007072180742397904, 3.1393941753776744e-05, 2.0462901375140063e-05, 0.004522331990301609, 0.000539271451998502, 0.0001313732354901731, 0.006195570807904005, 0.00017257021681871265, 7.375029781542253e-06, 0.0016005199868232012, 6.49317780698766e-06, 7.708074917900376e-06, 2.9949878808110952e-05, 6.522375770146027e-05, 0.001550215412862599, 3.604973608162254e-05, 4.4605019411392277e-07, 1.9028771930607036e-05, 0.00026783617795445025, 7.271899812621996e-05, 7.260347774717957e-05, 2.695044486245024e-06, 3.2488591386936605e-05, 0.0005929850740358233, 0.0006246539996936917, 8.568109478801489e-06, 1.1867665307363495e-05, 5.6961775953823235e-06, 0.0017213086830452085, 1.1051663022954017e-05, 4.6144563384586945e-05, 8.834384971123654e-06, 6.816963832534384e-06, 7.066460239002481e-05, 0.001541919307783246, 1.8624274161993526e-05, 0.003500899998471141, 8.836432971293107e-05, 0.0009130452526733279, 0.17712028324604034, 0.09427614510059357, 0.0002486585290171206, 0.00013264639710541815, 6.665812543360516e-05, 0.00088475551456213, 3.8286461290226725e-07, 1.2677381164394319e-05, 6.962897896301001e-05, 1.0584310984995682e-05, 5.303687430568971e-06, 0.005115383770316839, 1.3130702427588403e-06, 2.8680187824647874e-05, 1.8313443433726206e-05, 2.5623206965974532e-05, 4.521345999819459e-06, 0.0010276056127622724, 7.378776354016736e-05, 0.019413353875279427, 1.7400199112671544e-06, 0.00015067595813889056, 2.240733692815411e-06, 0.00020781725470442325, 0.003457006299868226, 3.3957455798372393e-06, 0.0020314825233072042, 9.023314305522945e-06, 3.357608648002497e-06, 0.000277730287052691, 0.000978577882051468, 0.00015551707474514842, 0.00026573881041258574, 9.435285755898803e-05, 2.2694994186167605e-05, 1.6480628062254254e-07, 3.4517102903919294e-05, 1.263202830159571e-05, 0.0005438763764686882, 8.70996245794231e-06, 0.04386834800243378, 0.0006701792008243501, 2.946042332041543e-05, 0.0007488338160328567, 4.9685764679452404e-05, 0.004985267296433449, 2.950394627987407e-05, 0.007820963859558105, 4.28969042332028e-06, 0.00014515407383441925, 8.685600914759561e-05, 7.1408476287615485e-06, 4.021987842861563e-05, 0.00010978544014506042, 2.987609377669287e-06, 8.19653632788686e-06, 0.0004586594586726278, 1.0593540537229273e-05, 0.012298870831727982, 0.012209189124405384, 2.5842920877039433e-05, 0.0003093622508458793, 0.0005243583000265062, 0.000920658465474844, 2.1312778699211776e-05, 0.0009885006584227085, 2.1471149011631496e-05, 7.88790566730313e-05, 0.004447180312126875, 8.241469913627952e-05, 5.857767064298969e-06, 1.8902775991591625e-05, 0.0002765425597317517, 1.0707441106205806e-05, 5.698046606994467e-06, 0.00016525988758075982, 6.184651738294633e-06, 0.05736962705850601, 1.4117213140707463e-05, 0.0008337290491908789, 1.3043163562542759e-06, 3.850962457363494e-05, 0.0012363119749352336, 0.0005267515662126243, 0.0007231798372231424, 1.8744014596450143e-05, 0.00018750604067463428, 5.785368921351619e-05, 1.5276149269993766e-06, 0.0032420253846794367, 1.368913672195049e-05, 5.205881461733952e-05, 0.001378583605401218, 0.0007576113566756248, 0.25401827692985535, 8.469382737530395e-05, 0.007979242131114006, 7.676621316932142e-05, 0.00011629497748799622, 0.002703920006752014, 0.0007836736040189862, 2.3669699658057652e-05, 8.293436258099973e-05, 0.00018391109188087285, 0.01705419085919857, 1.0891430974879768e-05, 0.00021296352497301996, 0.000666821317281574, 0.00016306816542055458, 2.353406307520345e-05, 0.0013195083010941744, 6.650252998952055e-06, 0.036240361630916595], [4.8229789939568946e-08, 3.7087045257067075e-07, 0.00020641402807086706, 0.0008743337821215391, 0.0005114185041747987, 2.178842441935558e-05, 8.186324703274295e-05, 0.00013653923815581948, 9.002265869639814e-06, 4.117681601201184e-05, 7.412657396344002e-06, 0.004184008110314608, 0.00045021867845207453, 3.491439565550536e-05, 4.169836756773293e-05, 3.271319656050764e-05, 4.987862121197395e-05, 0.0002889558672904968, 4.398525197757408e-05, 2.7909707569051534e-05, 0.000404241553042084, 7.831861876184121e-05, 0.00012876195251010358, 0.0001613064086996019, 7.395213469862938e-05, 4.450961114343954e-06, 6.136603042250499e-05, 0.00474718539044261, 9.904683793138247e-06, 1.0864678188227117e-05, 3.616803223849274e-05, 4.04752281610854e-05, 0.008160287514328957, 5.17065855092369e-05, 0.004119337536394596, 0.0013949453132227063, 0.00112911406904459, 0.0008617558632977307, 0.00037040762254036963, 0.0006193008157424629, 0.004932298790663481, 2.8339437108115817e-07, 0.0003768590686377138, 0.004241937771439552, 1.7925385691341944e-05, 0.00011147914483444765, 0.0016719535924494267, 6.493577870969602e-07, 0.0003066815552301705, 4.718183845398016e-06, 0.00025120200007222593, 0.0003830778587143868, 0.0002589893410913646, 5.957987468718784e-06, 0.00012706840061582625, 1.8549952073954046e-05, 0.0003108509408775717, 4.025139060104266e-05, 9.870332178252283e-06, 0.002074294025078416, 3.1903248327580513e-06, 2.710351509449538e-05, 0.0032296329736709595, 0.0003685049887280911, 0.0016916734166443348, 6.194392881297972e-06, 0.00020767940441146493, 8.769425221544225e-06, 6.061206363483507e-07, 0.0007079776842147112, 5.563430931943003e-06, 0.00017129656043834984, 7.0027322180976626e-06, 0.0006382914143614471, 0.0006812721840105951, 5.7600982472649775e-06, 5.690461421181681e-06, 4.134130549005022e-08, 2.9232334782136604e-05, 5.881169272470288e-05, 1.3036557902523782e-05, 0.0009698971407487988, 9.25607600947842e-05, 0.00011991327482974157, 1.80944953172002e-05, 1.2759686569552287e-06, 0.00014120804553385824, 0.0015050043584778905, 0.004549134057015181, 0.0002017242950387299, 2.695225020943326e-06, 4.3332325731171295e-05, 0.0001883789518615231, 4.306509708840167e-06, 3.836842370219529e-05, 1.701156907074619e-05, 1.0360229680372868e-05, 5.421375135483686e-06, 0.0003010481013916433, 0.0006016097613610327, 0.004761699121445417, 5.351715662982315e-05, 0.00014363035734277219, 6.578385409738985e-07, 0.0002771103463601321, 0.007229737937450409, 1.1422574061725754e-05, 0.0005285828956402838, 2.479215618222952e-05, 0.0005675151478499174, 0.0010553218889981508, 1.535571300337324e-06, 7.989290679688565e-06, 1.265584614884574e-05, 6.441748701035976e-05, 0.0060895886272192, 0.00013500102795660496, 0.00017839785141404718, 0.00021154551359359175, 1.6880117982509546e-05, 0.0030869438778609037, 0.000563620706088841, 0.0012678932398557663, 0.0002603287866804749, 1.9430462998570874e-06, 0.0006013737292960286, 1.807357693905942e-05, 7.759164873277768e-06, 0.39289698004722595, 0.0019157053902745247, 3.461342930677347e-05, 0.001465633395127952, 0.0033721206709742546, 0.00023978596436791122, 3.656816988950595e-05, 0.0023466991260647774, 0.0011967839673161507, 0.0009657376795075834, 7.9372273376066e-07, 0.00045705705997534096, 0.060166601091623306, 4.6019563342269976e-06, 3.553053466021083e-05, 1.1159801033500116e-05, 2.625172055559233e-05, 7.653662032680586e-05, 0.05367857962846756, 4.8291534767486155e-05, 0.015263666398823261, 6.869356639072066e-06, 0.0008205122430808842, 0.00010361332533648238, 3.414869570406154e-05, 0.00028555496828630567, 0.007027801591902971, 0.0004073874151799828, 0.0009148662793450058, 0.00016010180115699768, 0.0005411870661191642, 0.00043490948155522346, 0.0015420092968270183, 8.812404121272266e-05, 2.1335699784685858e-05, 3.1404397304868326e-05, 2.9642176741617732e-05, 0.0006658679340034723, 1.1583128070924431e-05, 0.0011661147000268102, 0.0002748704864643514, 0.0060270088724792, 0.0002502218121662736, 0.0034338762052357197, 0.0004971829475834966, 0.0002341457875445485, 5.362986121326685e-05, 0.0005552523070946336, 0.0001765630004229024, 2.011703145399224e-05, 2.995668546645902e-05, 3.798207330873993e-07, 0.00441852118819952, 0.0041539170779287815, 3.123659189441241e-05, 2.846673169187852e-06, 0.0010027914540842175, 0.00378325954079628, 0.00011928538151551038, 1.7500178728369065e-05, 0.0060178134590387344, 0.001887113437987864, 0.002886175410822034, 0.03932752087712288, 0.00011692421685438603, 0.00021437660325318575, 0.056730255484580994, 0.0016608302248641849, 0.2282974123954773, 5.001868785825536e-08, 1.2546410061986535e-06, 0.0007330169319175184], [0.00043296883814036846, 3.3002925192704424e-06, 6.0603570091188885e-06, 7.067445494612912e-06, 0.0006550237885676324, 1.2919547316414537e-06, 5.853251423104666e-05, 8.896932740753982e-06, 1.0209380008063818e-08, 0.00020038269576616585, 0.022721562534570694, 4.6427856432273984e-05, 0.0008412677561864257, 9.731589670991525e-05, 4.70398626930546e-05, 5.52461244751612e-07, 4.8456302465638146e-05, 1.8747478236491588e-07, 1.2487119420256931e-05, 0.00012328718730714172, 0.00013520725769922137, 7.0632609094900545e-06, 5.0212980568176135e-05, 3.5313967146066716e-06, 0.0008185880142264068, 2.5798333354032366e-06, 0.3465203046798706, 0.0010954440804198384, 7.24934579920955e-05, 0.021239668130874634, 5.9588824115053285e-06, 0.004087974317371845, 1.4597250810766127e-05, 0.00011592765804380178, 1.9019968021893874e-05, 9.924629921442829e-06, 1.7262505934922956e-05, 0.0001753161777742207, 3.961433321819641e-05, 0.025596972554922104, 0.0001274474197998643, 3.102665502296986e-08, 9.976490218832623e-06, 3.8668593333568424e-05, 0.00022372773673851043, 0.0005331385182216763, 1.6690460142854135e-06, 0.00015246403927449137, 6.162395948194899e-06, 0.00010232082422589883, 3.3383246773155406e-05, 0.0033188294619321823, 0.00163071951828897, 0.0078057716600596905, 7.292235295608407e-06, 1.5990137399057858e-05, 8.423442341154441e-05, 1.3667336133948993e-05, 1.1043915947084315e-05, 0.0003439372521825135, 0.056279491633176804, 0.0001998053485294804, 0.008375425823032856, 1.0156832104257774e-06, 0.03218608722090721, 0.056236326694488525, 0.00012993643758818507, 1.676306601439137e-05, 0.0004870739358011633, 5.070815518593008e-07, 0.0001880572090158239, 3.8684675018885173e-07, 0.009159436449408531, 6.85285558574833e-05, 8.113644980767276e-06, 1.3886365479720553e-07, 0.006081426050513983, 0.00010682139691198245, 1.5677631381549872e-05, 0.002948848530650139, 1.4384015685209306e-06, 0.002224748721346259, 0.004465540871024132, 3.9389960875269026e-05, 0.0033243841025978327, 0.01497490145266056, 1.0153069524676539e-05, 1.182414598588366e-05, 0.0011682758340612054, 0.0009818008402362466, 9.762798981682863e-06, 8.394851924720115e-09, 9.764805781742325e-07, 0.004094931297004223, 2.275877704960294e-05, 6.021894932928262e-06, 0.00031226465944200754, 5.107643232804548e-07, 0.08192460238933563, 2.988704181916546e-07, 5.61932974960655e-05, 0.00031743390718474984, 0.0006449734210036695, 3.59020759788109e-06, 3.610143903642893e-05, 0.002265313873067498, 0.00015097476716618985, 0.0007836472359485924, 0.00016329082427546382, 0.00018498535791877657, 7.707601980655454e-06, 2.9393356726359343e-06, 4.489076309255324e-05, 3.52507072420849e-06, 0.0075595988892018795, 0.0008904920541681349, 0.004219619557261467, 1.163817159977043e-06, 2.026835318247322e-06, 0.00072624278254807, 0.00016633773338980973, 0.0002029656752711162, 0.0018820316763594747, 4.933914624416502e-06, 2.2353312942868797e-06, 7.704684321652167e-06, 0.05322256684303284, 8.445619459962472e-05, 9.392204447067343e-06, 0.007565544918179512, 0.00142530573066324, 0.001335706328973174, 0.0006317687802948058, 5.0826871301978827e-05, 0.01963694579899311, 0.0011974386870861053, 2.7568560199142667e-06, 3.2588984595349757e-06, 1.3941326869826298e-05, 0.001276434981264174, 1.5391195120173506e-05, 4.3968371755909175e-05, 2.4573209884692915e-05, 0.000122146651847288, 0.003981188405305147, 2.7004582079825923e-05, 0.00010442383063491434, 0.017318101599812508, 1.0148756928174407e-06, 0.0002677025622688234, 0.00018573514535091817, 8.394508040510118e-05, 0.002192935673519969, 0.0737711489200592, 0.00010906723764492199, 9.377360402140766e-05, 0.0007316836272366345, 3.1797410429135198e-06, 3.706577990669757e-05, 0.0002726554812397808, 0.00020749263057950884, 1.197245092043886e-05, 0.002883827080950141, 3.580736347430502e-06, 1.332492411165731e-05, 5.272816451906692e-06, 0.0003430115757510066, 0.0012205910170450807, 7.936263136798516e-05, 0.0034566179383546114, 3.153998841298744e-05, 0.00014363105583470315, 0.00020915959612466395, 9.831167699303478e-05, 0.00018516213458497077, 5.171862085262546e-06, 2.0960327162811154e-07, 2.1121319150552154e-05, 3.5074604966212064e-05, 5.3170486353337765e-05, 4.779073151439661e-06, 2.5518254460621392e-06, 0.04431129992008209, 0.0010182621190324426, 0.0002941136190202087, 0.000178071903064847, 1.612806590856053e-05, 7.161936810007319e-05, 0.0032854455057531595, 0.0021800054237246513, 1.9890370822395198e-05, 2.53127291216515e-05, 2.2556159819941968e-05, 1.314350993197877e-05, 0.0016809845110401511, 0.0007492990116588771, 0.00028012788970954716, 1.804174462449737e-05, 0.004665057174861431, 0.00011696475849021226]], "dtype": "float32"}}, {"a": [{"__nd__": [[-24.884023666381836, 12.712523460388184, 2.3664767742156982, -5.8364996910095215, 13.127087593078613, -23.338754653930664, -6.507562637329102, 6.838550090789795, -7.123660564422607, 20.35786247253418, 19.822216033935547, 2.907978057861328, 2.3252365589141846, 17.31723403930664, 6.401284217834473, -10.270204544067383, -2.3566877841949463, -8.564983367919922, 6.2945356369018555, 1.9765586853027344, 14.247509002685547, 2.83461594581604, -7.3978776931762695, -10.912564277648926, 15.34644889831543, 13.009934425354004, -8.102127075195312, -14.504349708557129, -15.464309692382812, 10.172608375549316, -3.471325159072876, 18.361515045166016, -3.8470513820648193, -2.643744707107544, -2.1814656257629395, 6.497438907623291, 17.783313751220703, -9.055856704711914, 8.284931182861328, 14.4323148727417, -1.0597033500671387, -3.396122932434082, 11.475639343261719, -5.494483947753906, -2.2376627922058105, -8.902195930480957, -4.6006975173950195, 3.312944173812866, 4.297029495239258, -1.630609393119812, 1.4711568355560303, -12.469950675964355, 21.272823333740234, -10.337611198425293, 5.614455699920654, -7.520786285400391, 0.37422406673431396, -19.93338394165039, -6.674933910369873, -12.126757621765137, 3.4239790439605713, 1.1193727254867554, -9.124648094177246, -9.777054786682129], [4.168498516082764, -4.89048433303833, 5.877669811248779, 2.8670616149902344, -4.704962253570557, -2.772996425628662, -18.55337142944336, -0.24398274719715118, -2.370931386947632, -0.28309330344200134, 14.609280586242676, 1.381218671798706, 0.1467052549123764, -0.04843050241470337, -4.243849277496338, 14.192009925842285, -0.30360209941864014, 12.728117942810059, 7.123181343078613, -7.308418273925781, 2.2557361125946045, 11.261619567871094, -3.5031723976135254, 0.9766559600830078, -2.046121835708618, -1.81061589717865, 7.9309282302856445, -7.987302780151367, -3.057049036026001, 1.7547554969787598, -26.036956787109375, -3.0206871032714844, 1.0708011388778687, 3.945183515548706, -6.399324893951416, -9.367215156555176, 8.737476348876953, -6.839848041534424, 4.439889907836914, 0.8408973813056946, -7.450116157531738, -4.718258857727051, -7.844200611114502, 9.890769958496094, 1.9072932004928589, -17.149168014526367, 18.49818229675293, 13.952214241027832, 12.740238189697266, -9.030779838562012, -4.904702186584473, -5.739113807678223, 14.699971199035645, 0.9657791256904602, 3.010141134262085, -2.230684518814087, -13.677899360656738, 9.061466217041016, 22.518346786499023, -2.5367844104766846, -8.719770431518555, -6.497539043426514, -6.116310119628906, -14.506325721740723], [-4.987672805786133, -3.1907107830047607, -14.889933586120605, 4.394498348236084, -4.313671112060547, -11.356952667236328, 0.5058702230453491, 16.071292877197266, 14.137744903564453, 11.515028953552246, 11.496859550476074, 10.077383995056152, 7.654529094696045, 9.139176368713379, 20.591346740722656, -3.385434865951538, 8.686125755310059, -13.14400863647461, 7.103987693786621, 14.415802001953125, -2.8893818855285645, 17.66671371459961, -8.877084732055664, 16.07099151611328, -1.9032113552093506, 4.346386432647705, -3.227480411529541, 6.910181999206543, -1.0537527799606323, 11.784075736999512, 0.7668890357017517, 3.1113924980163574, -1.9492937326431274, 4.828405380249023, -4.100037097930908, -1.8328052759170532, -13.239274978637695, -4.121831893920898, -3.4313206672668457, -0.9832881093025208, 0.9886667132377625, -14.917804718017578, 8.44033145904541, 22.76103973388672, 0.8068447113037109, -19.036834716796875, 8.387502670288086, 0.543383777141571, -1.5569337606430054, 2.0719239711761475, 0.6883322596549988, -1.5539520978927612, 6.158396244049072, 3.1668801307678223, -2.5222463607788086, 7.693933486938477, 13.269499778747559, 13.585018157958984, -9.114623069763184, 1.3924379348754883, 7.003166198730469, 2.4632515907287598, -0.3339991569519043, 12.440108299255371], [4.326508522033691, 5.560208797454834, -14.038286209106445, -13.26983642578125, -4.0486836433410645, 2.7284696102142334, 10.128701210021973, -6.850218772888184, 16.72572898864746, 2.743242025375366, -3.7291290760040283, 8.919036865234375, 15.272994995117188, 1.6726239919662476, 1.4411964416503906, 3.768930196762085, 8.715775489807129, -1.2424784898757935, 0.6007866859436035, 2.6147220134735107, -0.5085574388504028, 4.4924821853637695, -6.089755535125732, -18.659582138061523, -4.422805309295654, -2.9949491024017334, 19.46485710144043, 11.226574897766113, 4.508309364318848, -4.249386310577393, -11.117000579833984, -6.057275772094727, 5.406627178192139, -0.9632446765899658, -9.897079467773438, 2.2427377700805664, -6.380945682525635, 4.725332260131836, 19.147207260131836, 9.012870788574219, -24.416826248168945, 11.425664901733398, -4.981647491455078, 3.010955333709717, 3.6404612064361572, 2.660177230834961, -17.452299118041992, 12.830940246582031, -9.19703483581543, -8.732675552368164, 2.6597516536712646, 7.348083019256592, -5.389246463775635, -15.514405250549316, 4.526386737823486, -2.568621873855591, 8.794794082641602, 12.609223365783691, -11.997488021850586, -12.311302185058594, -1.3068617582321167, -8.257227897644043, 17.207321166992188, 1.514235258102417], [1.6462461948394775, 2.930680751800537, 15.34352970123291, 4.833763599395752, 8.818994522094727, -6.202971458435059, 8.151737213134766, -3.1839561462402344, -4.853025913238525, -5.099769592285156, 12.492151260375977, -2.668288469314575, 4.560995101928711, 1.5613017082214355, 5.631207466125488, -5.311193943023682, -12.77837085723877, -2.798882484436035, 1.4406472444534302, -8.11825180053711, 0.05137673765420914, 7.028846740722656, -19.689104080200195, 15.269493103027344, 10.340177536010742, 3.4539575576782227, 12.220186233520508, -19.616413116455078, 2.2290658950805664, 4.445153713226318, 5.473728179931641, 5.067657947540283, 18.886648178100586, 7.577681064605713, -5.519406795501709, -13.881027221679688, -9.055668830871582, -3.1972689628601074, 2.6867170333862305, 0.403048574924469, 3.0239434242248535, -5.240545272827148, -3.5278570652008057, 4.342823505401611, -1.936848759651184, -6.311365604400635, -4.257635593414307, 4.958215713500977, -4.97341775894165, -2.868142604827881, 18.406450271606445, -1.008720874786377, 2.1875314712524414, 11.325687408447266, 17.518203735351562, 7.082569599151611, -12.697086334228516, 24.092159271240234, -8.90012264251709, -11.505026817321777, -8.031428337097168, 14.696014404296875, -0.3683125376701355, -18.970983505249023], [18.848323822021484, -15.590377807617188, 18.819469451904297, -25.341678619384766, -1.2436652183532715, 0.31642699241638184, -5.916224956512451, -0.6840634346008301, 10.789896965026855, 2.033874750137329, 7.27652645111084, 1.2938220500946045, 2.0159621238708496, -5.6931939125061035, -0.5159622430801392, 9.151204109191895, 11.198866844177246, -11.534971237182617, -6.1344146728515625, 1.8649027347564697, -1.1764484643936157, -8.013016700744629, -20.479721069335938, 2.912433624267578, -6.888991832733154, 20.69577407836914, 11.90317153930664, 4.465017795562744, -12.975154876708984, -7.133692741394043, 11.335148811340332, -7.715695858001709, -2.312936305999756, 4.5431904792785645, 6.285086631774902, -3.393444538116455, 5.296193599700928, -16.568382263183594, 0.11498758941888809, -1.064680576324463, 17.003150939941406, 4.403181552886963, -22.310152053833008, 15.621528625488281, 27.43752670288086, 5.855227947235107, 7.183528900146484, 16.207141876220703, 0.0897422507405281, -0.29134103655815125, 2.6606624126434326, -33.15687942504883, -5.538376808166504, 6.85357141494751, 6.167287349700928, -15.306259155273438, -2.442587375640869, -5.6107892990112305, -7.272836208343506, 9.382730484008789, -8.944746971130371, -2.306295871734619, 6.369784355163574, -9.428930282592773], [-0.4173329770565033, -10.232171058654785, -0.548608660697937, -1.4848568439483643, 5.997647285461426, -5.031332969665527, -3.3171281814575195, -16.6251163482666, -6.899007797241211, -10.965943336486816, -13.62387752532959, -4.21951150894165, -0.6485419869422913, -1.2269333600997925, 10.59192180633545, -6.05718994140625, -7.943202018737793, -22.09803009033203, 10.876752853393555, 6.664713382720947, 6.04664421081543, -8.0855131149292, -6.490434646606445, -6.097248554229736, -13.471526145935059, 3.4098830223083496, -1.3592413663864136, 11.963529586791992, 1.5528138875961304, -2.3884451389312744, 6.3656392097473145, 6.885055065155029, -11.218534469604492, -16.5447998046875, -12.06605052947998, 8.999324798583984, 6.126236915588379, 13.291619300842285, 0.4743396043777466, 3.6470725536346436, -4.141066551208496, 19.609207153320312, 6.807933330535889, 15.471702575683594, 4.811099052429199, -7.757075309753418, 9.918335914611816, -15.370366096496582, 8.0184907913208, -1.2980023622512817, 3.9386024475097656, -12.128005027770996, 0.660679042339325, -6.438279151916504, -8.654875755310059, 14.004254341125488, 5.100016117095947, -3.838479995727539, -6.729388236999512, -3.897272825241089, 1.557422161102295, 3.1759250164031982, 0.8062140941619873, -17.380630493164062], [0.056850090622901917, 2.2829153537750244, 7.334649085998535, -0.42745286226272583, -1.71726393699646, 0.5441047549247742, -5.505432605743408, 20.618999481201172, -7.757776737213135, -9.854361534118652, 2.597726345062256, 2.4445722103118896, 6.244190216064453, -1.9455467462539673, -5.826284408569336, -8.96573257446289, -0.15545429289340973, 2.473841905593872, -0.6707322597503662, -12.44528865814209, -13.2836275100708, 10.80626106262207, 0.061012089252471924, -1.9795809984207153, 17.692663192749023, 8.981080055236816, -13.44575023651123, -3.243076801300049, 9.398655891418457, 6.0841264724731445, 2.9013121128082275, 15.718463897705078, -13.425052642822266, 8.164067268371582, -5.165759086608887, 12.175153732299805, -0.4505247175693512, -15.940650939941406, 2.653762102127075, -24.961441040039062, 11.232608795166016, 3.5554914474487305, 12.124110221862793, 3.327166795730591, 2.2596073150634766, 14.966495513916016, 8.50851058959961, -14.04383373260498, -4.749978065490723, 15.383500099182129, -11.462986946105957, -12.308014869689941, -8.093770027160645, -14.77194881439209, 1.3527612686157227, -2.111785650253296, 9.937116622924805, -3.0103487968444824, 15.887796401977539, -3.072322368621826, 8.907211303710938, 11.660090446472168, 3.6403026580810547, -2.1267752647399902]], "dtype": "float32"}], "e": {"__nd__": [[5.160283856468235e-21, 0.00010981097148032859, 3.527072856002178e-09, 9.658443749643864e-13, 0.00016622206021565944, 2.4197682480075643e-20, 4.937059875503857e-13, 3.0875338552505127e-07, 2.666246401179101e-13, 0.22960127890110016, 0.13438351452350616, 6.061578794458455e-09, 3.384579283505218e-09, 0.010976050980389118, 1.993927583043842e-07, 1.1464993281736901e-14, 3.134516604608173e-11, 6.308729851457057e-14, 1.7920456230058335e-07, 2.3882265054453455e-09, 0.0005096610984764993, 5.632807553723751e-09, 2.0267953439699615e-13, 6.03116088158679e-15, 0.0015294842887669802, 0.00014784603263251483, 1.0022080279385726e-13, 1.661530564911175e-16, 6.362130641276054e-17, 8.661131687404122e-06, 1.028228630711725e-11, 0.03118685819208622, 7.0617717402854385e-12, 2.3523552897253275e-11, 3.7348114184654335e-11, 2.1951723283564206e-07, 0.017492907121777534, 3.861520884952327e-14, 1.311495680056396e-06, 0.0006131146219559014, 1.1466831650874809e-10, 1.1085363529894643e-11, 3.187673428328708e-05, 1.3596993539768953e-12, 3.530710793175906e-11, 4.502905074635008e-14, 3.323608315647375e-12, 9.087835373122743e-09, 2.431325718532662e-08, 6.478914627017218e-11, 1.4407292026064056e-09, 1.2706788051641436e-15, 0.573240339756012, 1.0717636630616645e-14, 9.078086549152431e-08, 1.7923836514392028e-13, 4.810500331409173e-10, 7.28967700665805e-19, 4.176185493341006e-13, 1.790946558400232e-15, 1.0155050134130761e-08, 1.0134553196650131e-09, 3.604808576748893e-14, 1.8773551845583693e-14], [1.0530936123132051e-08, 1.2251822820164016e-12, 5.817691217657739e-08, 2.8658926343183566e-09, 1.4749318303156067e-12, 1.0181543975973373e-11, 1.427200161503707e-18, 1.2768817947428346e-10, 1.5220481125455265e-11, 1.2279066652354231e-10, 0.0003604465164244175, 6.485839643133318e-10, 1.8872277351977829e-10, 1.552663692061529e-10, 2.3390029062902995e-12, 0.00023747750674374402, 1.2029792439971487e-10, 5.493666321854107e-05, 2.021479019731487e-07, 1.0917046482311968e-13, 1.5551214760378684e-09, 1.2675645848503336e-05, 4.9057190601442535e-12, 4.327800062764453e-10, 2.1061610788741802e-11, 2.6654499787892583e-11, 4.533872015599627e-07, 5.536934097522851e-14, 7.663928519585639e-12, 9.423045677081632e-10, 8.024230523906694e-22, 7.947730147617182e-12, 4.755035254433437e-10, 8.423294950432592e-09, 2.709689840129509e-13, 1.3930945901518242e-14, 1.0156607004319085e-06, 1.744226372493371e-13, 1.3814354105079474e-08, 3.7783975948002535e-10, 9.474714713969198e-14, 1.4554479583345215e-12, 6.388781241236166e-14, 3.218234496671357e-06, 1.0975842457128238e-09, 5.811949654809884e-18, 0.01761040650308132, 0.00018684458336792886, 5.560656063607894e-05, 1.950264223119641e-14, 1.2078843784554816e-12, 5.243786633211955e-13, 0.00039466374437324703, 4.2809789047026925e-10, 3.3067293347954774e-09, 1.7512050837220627e-11, 1.8701404861389242e-16, 1.4042863085705903e-06, 0.9810804724693298, 1.2894339242175423e-11, 2.6617219238693653e-14, 2.4562090642638035e-13, 3.596094776125991e-13, 8.167577397881655e-17], [7.91243264484931e-13, 4.772226450816541e-12, 3.9610688450069504e-17, 9.395835220971094e-09, 1.5524773307185047e-12, 1.355710431035753e-15, 1.923648601520611e-10, 0.0011068853782489896, 0.0001600934047019109, 1.1623645150393713e-05, 1.1414357686589938e-05, 2.7604576189332874e-06, 2.4476469207002083e-07, 1.0802460792547208e-06, 0.1016569510102272, 3.927833319011809e-12, 6.866972626085044e-07, 2.2701700596276927e-16, 1.4114044688540162e-07, 0.0002114131930284202, 6.4503966404338975e-12, 0.005457391031086445, 1.618675896444821e-14, 0.0011065518483519554, 1.729316516374535e-11, 8.954478047940029e-09, 4.599939353033822e-12, 1.1627402329850156e-07, 4.0438007109511176e-11, 1.5212054677249398e-05, 2.4973806356243244e-10, 2.6042916712043507e-09, 1.6514352385588182e-11, 1.450035469474642e-08, 1.9222308206973127e-12, 1.855459362543055e-11, 2.0638853761890545e-16, 1.8807887105370957e-12, 3.751676486835054e-12, 4.339026707378402e-11, 3.117462399337967e-10, 3.852197648888436e-17, 5.370542908167408e-07, 0.8900671601295471, 2.599183923646109e-10, 6.263780727599877e-19, 5.094187827126007e-07, 1.9971824194442434e-10, 2.4448984503600002e-11, 9.209882301242089e-10, 2.3086996203680599e-10, 2.452202677027948e-11, 5.482594644945493e-08, 2.752879479928083e-09, 9.311738012607318e-12, 2.5460192887294397e-07, 6.718918302794918e-05, 9.211433643940836e-05, 1.2764326576231947e-14, 4.66827576595108e-10, 1.2760429513036797e-07, 1.362088108969317e-09, 8.305669346730937e-11, 2.9315628125914373e-05], [1.390745296703244e-07, 4.775699835590785e-07, 1.4706813893401315e-15, 3.1714016805614803e-15, 3.2058859039674203e-11, 2.8133788632089818e-08, 4.603709385264665e-05, 1.94651174519056e-12, 0.03374120965600014, 2.8552479491850136e-08, 4.412942927745078e-11, 1.3732744264416397e-05, 0.007893076166510582, 9.78769509885069e-09, 7.765554421723664e-09, 7.963326709159446e-08, 1.1206810086150654e-05, 5.304773842418342e-10, 3.3510967334393627e-09, 2.5108910506332904e-08, 1.105108116128406e-09, 1.6418331938439223e-07, 4.164102222758981e-12, 1.447155790383972e-17, 2.20530520927964e-11, 9.195574857123745e-11, 0.522094190120697, 0.00013800930173601955, 1.6680256464951526e-07, 2.622913691796569e-11, 2.7303405711462045e-14, 4.301573854059715e-12, 4.0957871760838316e-07, 7.013530312072191e-10, 9.247446965078218e-14, 1.730923493425962e-08, 3.1121429218405883e-12, 2.0723091154195572e-07, 0.3800102174282074, 1.5083734979270957e-05, 4.5727339130801245e-20, 0.00016841162869241089, 1.2611502120396523e-11, 3.7317310130902115e-08, 7.003271207395301e-08, 2.627659689835582e-08, 4.839846998937566e-17, 0.0006865550531074405, 1.8622901614903653e-13, 2.962893602340144e-13, 2.6265423613835992e-08, 2.8543086045829114e-06, 8.389749832360405e-12, 3.360839858061076e-16, 1.6984525075258716e-07, 1.408413524695007e-10, 1.2128284652135335e-05, 0.0005500282277353108, 1.1319467092167981e-14, 8.27062366753591e-15, 4.974001766022695e-10, 4.766512163266456e-13, 0.054615188390016556, 8.35396285481238e-09], [1.7670437335581823e-10, 6.383670259069163e-10, 0.00015700086078140885, 4.281228260794023e-09, 2.3032040985526692e-07, 6.892473110265904e-14, 1.1818061551593928e-07, 1.4109680221072174e-12, 2.6585789234152835e-13, 2.0772557393806956e-13, 9.069098268810194e-06, 2.36302600750693e-12, 3.2591680465543504e-09, 1.623142592555027e-10, 9.503720477255229e-09, 1.6813949592440341e-13, 9.609841572995234e-17, 2.0737335459930506e-12, 1.43865572233004e-10, 1.0152659725567396e-14, 3.585949939766131e-11, 3.844865048563406e-08, 9.581238851315402e-20, 0.00014579691924154758, 1.0543233202042757e-06, 1.0772737146780287e-09, 6.909573130542412e-06, 1.0303666732811392e-19, 3.164917217191032e-10, 2.9026703263212994e-09, 8.118984595739676e-09, 5.409379166820827e-09, 0.005428237374871969, 6.656357243173261e-08, 1.3653488062370656e-13, 3.190349264961077e-17, 3.976172130670764e-15, 1.3923079269367866e-12, 5.00171071227129e-10, 5.097226113415232e-11, 7.007667224279146e-10, 1.8044791464007226e-13, 1.000375455216529e-12, 2.620329064839666e-09, 4.910533351470958e-12, 6.184444368293254e-14, 4.821968020186918e-13, 4.848612622510018e-09, 2.3570243521710277e-13, 1.934964775213155e-12, 0.00335823860950768, 1.2422530885902727e-11, 3.0361568814640805e-10, 2.8247193313291064e-06, 0.00138150027487427, 4.057069347140896e-08, 1.042358811065591e-16, 0.9894266128540039, 4.645336629331134e-15, 3.43337870087275e-16, 1.1073537005163326e-14, 8.216554124373943e-05, 2.3568693871545143e-11, 1.9647130578343865e-19], [0.00018580745381768793, 2.053643503983904e-19, 0.00018052270752377808, 1.1956083164612765e-23, 3.493196129491366e-13, 1.6625041187462442e-12, 3.265547910341138e-15, 6.113013198937034e-13, 5.879403275343975e-08, 9.260631324281565e-12, 1.7518455575071812e-09, 4.418140766432321e-12, 9.096232447824981e-12, 4.08147187854163e-15, 7.232033439989738e-13, 1.1419764689435397e-08, 8.850067700905129e-08, 1.1851316075428972e-17, 2.6254133483655413e-15, 7.820904514288518e-12, 3.7360690713014733e-13, 4.011729091488842e-16, 1.5456468522534821e-21, 2.2294307025494398e-11, 1.234493345899203e-15, 0.0011786927934736013, 1.7898713622344076e-07, 1.053107046011803e-10, 2.8073848245559717e-18, 9.665322495738186e-16, 1.014220316619685e-07, 5.400781598064721e-16, 1.1990697334780115e-13, 1.1387344539537381e-10, 6.500083804539258e-10, 4.069922160789344e-14, 2.417953615108104e-10, 7.722948524646615e-20, 1.3591859842482235e-12, 4.1778746803257383e-13, 2.9357208404690027e-05, 9.899593644835392e-11, 2.478354082295521e-22, 7.373671905952506e-06, 0.9984045624732971, 4.2289591273281246e-10, 1.596272669601717e-09, 1.3243793546280358e-05, 1.3253020642728375e-12, 9.0534058114905e-13, 1.7332085419652365e-11, 4.824922511666512e-27, 4.76490252961613e-15, 1.1476465333615238e-09, 5.777756051372762e-10, 2.7284513730970404e-19, 1.0532660897968893e-13, 4.432052039146607e-15, 8.40984708494431e-16, 1.439488439558545e-08, 1.580103170109251e-16, 1.2070596258628719e-13, 7.074613117552531e-10, 9.736627185075931e-17], [1.963350149125631e-09, 1.0726756126139964e-13, 1.721811693222719e-09, 6.751153525108577e-10, 1.199467646983976e-06, 1.9460898170731156e-11, 1.0805213668252378e-10, 1.7949317690077248e-16, 3.0063698926163784e-12, 5.149856642473402e-14, 3.609675373953932e-15, 4.3826123286017094e-11, 1.5580630119416128e-09, 8.73762506792275e-10, 0.00011864695261465386, 6.9765265613142e-12, 1.0581670088771755e-12, 7.536867525010334e-19, 0.00015774556959513575, 2.3371760562440613e-06, 1.2597015484061558e-06, 9.17802617760033e-13, 4.5235924381326065e-12, 6.702583366796633e-12, 4.2037177910717385e-15, 9.018556568207714e-08, 7.654773481746702e-10, 0.000467668694909662, 1.408063265984083e-08, 2.7349852937952335e-10, 1.7330264654447092e-06, 2.9132988856872544e-06, 4.0003299338429305e-14, 1.9450383157087016e-16, 1.714051298085887e-14, 2.413239963061642e-05, 1.3640626548294676e-06, 0.0017649009823799133, 4.789022511886287e-09, 1.1432652513576613e-07, 4.740243614298478e-11, 0.9781689643859863, 2.697065610846039e-06, 0.015614155679941177, 3.6616529541788623e-07, 1.274641308300628e-12, 6.0495349316624925e-05, 6.294752346695527e-16, 9.049606887856498e-06, 8.138197338247721e-10, 1.5302300937491964e-07, 1.6110808908199707e-14, 5.769956956669375e-09, 4.765788024635453e-12, 5.19371600848223e-13, 0.0035992648918181658, 4.888235025646281e-07, 6.415237091550452e-11, 3.562106122220743e-12, 6.048945372372216e-11, 1.4145668636444952e-08, 7.13723196099636e-08, 6.67386501618239e-09, 8.432017113547507e-17], [1.0883655088278488e-09, 1.0081900647662678e-08, 1.5757327673782129e-06, 6.705711541599157e-10, 1.8462333051250113e-10, 1.7716870193140721e-09, 4.179325288622415e-12, 0.9264087677001953, 4.3946625310174736e-13, 5.3999495576006334e-14, 1.3812217147801675e-08, 1.1850843861793692e-08, 5.295443088471075e-07, 1.4694154226724265e-10, 3.0322276807492887e-12, 1.3131573465326868e-13, 8.801807616087842e-10, 1.2202846733089245e-08, 5.257624335897049e-10, 4.047290287471656e-15, 1.750165945396437e-15, 5.07206168549601e-05, 1.0929045446417263e-09, 1.4202457265799495e-10, 0.04964905232191086, 8.175577931979205e-06, 1.4882277827216227e-15, 4.014530721740961e-11, 1.2412788237270433e-05, 4.512191082994832e-07, 1.8711503457780054e-08, 0.006894886028021574, 1.5193529612805476e-15, 3.611555712268455e-06, 5.869807678110517e-12, 0.00019938251352868974, 6.552760556388648e-10, 1.2278581662464457e-16, 1.4608295018092576e-08, 1.4841195386656862e-20, 7.768639625282958e-05, 3.5992805180740106e-08, 0.0001894606975838542, 2.864543624525595e-08, 9.84963044459164e-09, 0.0032505090348422527, 5.096642325952416e-06, 8.183235066609447e-16, 8.896028011462498e-12, 0.0049323528073728085, 1.0808698601677113e-14, 4.642826447191946e-15, 3.1405498861214465e-13, 3.951015093318634e-16, 3.97723542988615e-09, 1.244365582797613e-10, 2.126766776200384e-05, 5.066486466476228e-11, 0.008167088963091373, 4.762029659488576e-11, 7.5934249252895825e-06, 0.00011912362242583185, 3.917854485280259e-08, 1.225851919972598e-10]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(softmax, json.loads(_TESTS), 'cuda', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")